# Multilingual Text Transliteration with Hugging Face and MT5

This notebook demonstrates how to build and train a sequence-to-sequence model using Hugging Face Transformers for transliterating Romanized text to Amharic text. It leverages the MT5 model and is set up to optionally run on TPUs.

## 1. Setup and Imports

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
! pip install evaluate gcsfs google-cloud-storage jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [ ]:
import evaluate
from datasets import Dataset, DatasetDict

In [ ]:
import os

# Before torch is first used: reduces CUDA fragmentation (PyTorch 2.0+).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

### TPU/XLA Configuration (Optional)

In [ ]:
# TPU/XLA (optional)
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    IS_TPU = True
except ImportError:
    IS_TPU = False
    print("torch_xla not available. Training on CPU/GPU.")

torch_xla not available. Training on CPU/GPU.


## 2. Configuration

In [ ]:
CLEAN_CSV_PATH = "/content/drive/MyDrive/datasets/merged_normalized.csv"
MODEL_CHECKPOINT = "google/mt5-small"
GCS_BUCKET_NAME = "geezify-transliteration"
# Used only when the bucket is created by this notebook (existing buckets keep their location).
GCS_BUCKET_LOCATION = "US"

# GCS: Colab runs ``authenticate_user()`` once (browser). Locally: ``gcloud auth application-default login`` or set ``GOOGLE_APPLICATION_CREDENTIALS``.
import gcsfs
import google.auth

# Full control allows creating the bucket from Colab plus read/write/list objects.
_GCS_SCOPES = ("https://www.googleapis.com/auth/devstorage.full_control",)


def _gc_credentials():
    try:
        from google.colab import auth as _colab_auth  # type: ignore

        _colab_auth.authenticate_user()
    except ImportError:
        pass
    credentials, project = google.auth.default(scopes=_GCS_SCOPES)
    return credentials, project


def _gcs_fs():
    """Credentials that work on Colab GPU/CPU (metadata 404) and on GCE/TPU VMs."""
    credentials, project = _gc_credentials()
    return gcsfs.GCSFileSystem(project=project, token=credentials)


def ensure_gcs_bucket_exists() -> None:
    """Create ``GCS_BUCKET_NAME`` if missing (needs bucket create permission on the GCP project)."""
    from google.cloud import storage

    credentials, project = _gc_credentials()
    client = storage.Client(project=project, credentials=credentials)
    bucket = client.bucket(GCS_BUCKET_NAME)
    if bucket.exists():
        print(f"GCS bucket gs://{GCS_BUCKET_NAME} found.")
        return
    try:
        client.create_bucket(GCS_BUCKET_NAME, location=GCS_BUCKET_LOCATION)
        print(
            f"Created GCS bucket gs://{GCS_BUCKET_NAME} (location={GCS_BUCKET_LOCATION}). "
            "If uploads still fail, add your account under bucket Permissions."
        )
    except Exception as exc:
        raise RuntimeError(
            f"Could not create gs://{GCS_BUCKET_NAME}. In Cloud Console create the bucket manually, "
            "or grant this identity roles/storage.admin (or storage.buckets.create) on the project."
        ) from exc


GCS_MODELS_ROOT = f"gs://{GCS_BUCKET_NAME}/models"
# Upload the finished run here (upload cell). Presence of final/config.json switches new runs to checkpoints1, ...
GCS_FINAL_MODEL_DIR = f"{GCS_MODELS_ROOT}/final"
RESUME_TRAINING = True  # Only resumes when checkpoint-* exists (never pass True into train() on an empty dir).
# Roughly how many times to write a checkpoint over the whole run (pick 10–20 to limit GCS cost).
NUM_CHECKPOINT_SAVES = 15


def resolve_training_output_dir():
    """HF checkpoints path on GCS.

    Until ``models/final/config.json`` exists → ``.../models/checkpoints`` (resume if checkpoints present).
    After final is published → ``.../models/checkpoints1``, ``checkpoints2``, …; resume the latest that has HF checkpoints.
    """
    ensure_gcs_bucket_exists()
    fs = _gcs_fs()
    models_prefix = f"{GCS_BUCKET_NAME}/models"
    final_marker = f"{models_prefix}/final/config.json"

    def _forbidden(exc: BaseException) -> bool:
        msg = str(exc).lower()
        return "403" in msg or "forbidden" in msg or "permission" in msg

    try:
        fs.exists(final_marker)
    except Exception as exc:
        if _forbidden(exc):
            raise RuntimeError(
                f"GCS IAM: your account needs roles/storage.objectAdmin (or storage.admin) on "
                f"gs://{GCS_BUCKET_NAME} so it can list/read/write objects (storage.objects.list). "
                "Console → Cloud Storage → open the bucket → Permissions → Grant access."
            ) from exc
        raise RuntimeError(
            f"Cannot access gs://{GCS_BUCKET_NAME}/models — check auth and that the bucket exists."
        ) from exc

    has_final = fs.exists(final_marker)

    def has_hf_checkpoint(prefix: str) -> bool:
        try:
            return bool(fs.glob(f"{prefix}/checkpoint-*/trainer_state.json"))
        except Exception as exc:
            if _forbidden(exc):
                raise RuntimeError(
                    "GCS IAM: listing checkpoints requires storage.objects.list on the bucket. "
                    "Grant Storage Object Admin on gs://"
                    + GCS_BUCKET_NAME
                    + " to your Google user (same as Colab auth)."
                ) from exc
            raise

    if not has_final:
        out = f"gs://{models_prefix}/checkpoints"
        print(f"OUTPUT_DIR (no published final yet): {out}")
        return out

    last_k = 0
    for k in range(1, 128):
        if has_hf_checkpoint(f"{models_prefix}/checkpoints{k}"):
            last_k = k
    if last_k:
        out = f"gs://{models_prefix}/checkpoints{last_k}"
        print(f"OUTPUT_DIR (resume numbered run after final): {out}")
        return out
    out = f"gs://{models_prefix}/checkpoints1"
    print(f"OUTPUT_DIR (new numbered run; models/final exists): {out}")
    return out


OUTPUT_DIR = resolve_training_output_dir()
TEST_SIZE = 0.1
RANDOM_STATE = 42
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128
# Micro-batch for ~15GB GPUs (mT5 seq2seq + eval often OOMs at 8). Effective batch ≈ BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS.
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
EVAL_BATCH_SIZE = 1  # eval merges preds on GPU; use 1 + eval_accumulation_steps in training args
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

USE_SUBSET = True
SUBSET_ROWS = 30000
NUM_EPOCHS = 3 if USE_SUBSET else 10  # Faster iteration for smoke test

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Data Loading and Preparation

In [ ]:
print("Loading dataset...")

assert os.path.exists(CLEAN_CSV_PATH)

df = pd.read_csv(CLEAN_CSV_PATH)

if USE_SUBSET:
    df = df.head(min(SUBSET_ROWS, len(df))).copy()
    print(f"Using smoke-test subset with {len(df)} rows")

# Expected format:
# romanized_text,amharic_text

required_columns = {"romanized_text", "amharic_text"}

if not required_columns.issubset(df.columns):
    raise ValueError(
        f"Dataset must contain columns: {required_columns}"
    )

# Remove missing values
df = df.dropna(subset=["romanized_text", "amharic_text"])

# Convert to string
df["romanized_text"] = df["romanized_text"].astype(str)
df["amharic_text"] = df["amharic_text"].astype(str)

# Remove empty rows
df = df[
    (df["romanized_text"].str.strip() != "")
    & (df["amharic_text"].str.strip() != "")
]

df["source"] = df["romanized_text"]
df["target"] = df["amharic_text"]

train_ready_df = df[["source", "target"]].copy()

print(f"Dataset size after cleaning: {len(train_ready_df)}")

Loading dataset...
Using smoke-test subset with 30000 rows
Dataset size after cleaning: 30000


### Train-Test Split

In [ ]:
train_df, test_df = train_test_split(
    train_ready_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

raw_datasets = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df),
        "test": Dataset.from_pandas(test_df),
    }
)

## 4. Tokenizer and Model Initialization

In [ ]:
print("Loading tokenizer/model...")

# use_fast=False helps some multilingual tokenizers
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    use_fast=True,
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

# mT5 may leave pad_token unset; labels/collator need a real pad id.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

# Lower VRAM at the cost of slower steps (required with TrainingArguments.gradient_checkpointing).
model.config.use_cache = False
model.gradient_checkpointing_enable()

Loading tokenizer/model...


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


## 5. Preprocessing Data for Training

In [ ]:
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["source"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        text_target=examples["target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    # For LengthGroupedSampler: batch similar encoder lengths → less padding waste.
    model_inputs["length"] = [len(ids) for ids in model_inputs["input_ids"]]

    return model_inputs

In [ ]:
print("Tokenizing dataset...")

# num_proc speeds up preprocessing on TPU VM CPUs
tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=raw_datasets["train"].column_names,
)

Tokenizing dataset...


Map (num_proc=4):   0%|          | 0/27000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/3000 [00:00<?, ? examples/s]

## 6. Metrics Setup

In [ ]:
!pip install -q sacrebleu jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.8 MB/s eta 0:00:00


In [ ]:
import jiwer

metric = evaluate.load("sacrebleu")


def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels


def _transliteration_metrics(refs_flat, hyps):
    """One string reference per hypothesis; Amharic/Roman transliteration scores."""
    if not hyps:
        return {"cer": 0.0, "word_accuracy": 1.0, "mean_char_edit_distance": 0.0}

    cer = jiwer.cer(refs_flat, hyps)
    wer = jiwer.wer(refs_flat, hyps)
    word_accuracy = 1.0 - wer

    char_edits = []
    for r, h in zip(refs_flat, hyps):
        m = jiwer.compute_measures(r, h)
        char_edits.append(m["substitutions"] + m["deletions"] + m["insertions"])
    mean_char_edit = float(np.mean(char_edits))

    return {
        "cer": float(cer),
        "word_accuracy": float(word_accuracy),
        "mean_char_edit_distance": mean_char_edit,
    }

In [ ]:
def make_compute_metrics(tokenizer_ref):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds

        if isinstance(preds, tuple):
            preds = preds[0]

        # Clip predictions to valid token ID range to prevent OverflowError
        # This can happen if the model generates out-of-range values, especially in early training
        preds = np.clip(preds, 0, tokenizer_ref.vocab_size - 1)

        decoded_preds = tokenizer_ref.batch_decode(
            preds,
            skip_special_tokens=True,
        )

        labels = np.where(labels != -100, labels, tokenizer_ref.pad_token_id)

        decoded_labels = tokenizer_ref.batch_decode(
            labels,
            skip_special_tokens=True,
        )

        decoded_preds, decoded_labels = postprocess_text(
            decoded_preds,
            decoded_labels,
        )

        result = metric.compute(
            predictions=decoded_preds,
            references=decoded_labels,
        )

        ref_flat = [row[0] for row in decoded_labels]
        tm = _transliteration_metrics(ref_flat, decoded_preds)

        return {
            "bleu": result["score"],
            "cer": tm["cer"],
            "word_accuracy": tm["word_accuracy"],
            "mean_char_edit_distance": tm["mean_char_edit_distance"],
        }

    return compute_metrics

## 7. Data Collator

In [ ]:
# pad_to_multiple_of helps Tensor Cores; drop "length" so it is not passed to the model.
_pad_mul = None if IS_TPU else 8
_base_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    pad_to_multiple_of=_pad_mul,
)


def data_collator(features):
    features = [{k: v for k, v in row.items() if k != "length"} for row in features]
    return _base_collator(features)

## 8. Training Arguments

In [ ]:
import math

import torch

# Baseline step if eval and save were tied. Eval runs 2× more often; checkpoints 2× less often (half frequency).
_n_train = len(tokenized_datasets["train"])
_steps_per_epoch = max(1, math.ceil(_n_train / (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)))
_total_steps = max(1, _steps_per_epoch * NUM_EPOCHS)
_base = max(1, math.ceil(_total_steps / NUM_CHECKPOINT_SAVES))
_eval_steps = max(1, _base // 2)
_save_steps = max(1, _base * 2)

_cuda_ok = torch.cuda.is_available() and not IS_TPU
_use_bf16 = IS_TPU or (_cuda_ok and torch.cuda.is_bf16_supported())
_use_fp16 = _cuda_ok and not _use_bf16

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    eval_accumulation_steps=1,  # move eval predictions to CPU each step (avoids huge GPU pad/concat OOM)
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    predict_with_generate=False,
    generation_max_length=MAX_TARGET_LENGTH,
    logging_steps=20,
    logging_nan_inf_filter=False,
    eval_strategy="steps",
    eval_steps=_eval_steps,
    save_strategy="steps",
    save_steps=_save_steps,
    save_total_limit=NUM_CHECKPOINT_SAVES + 2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_bleu",
    greater_is_better=True,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    bf16=_use_bf16,
    fp16=_use_fp16,
    report_to="none",
    dataloader_num_workers=2,
    remove_unused_columns=False,  # keep "length" for group_by_length
    group_by_length=True,
    length_column_name="length",
    push_to_hub=False,
)

## 9. Initialize Trainer

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(tokenizer),
)

## 10. Start Training

In [ ]:
import os

from transformers.trainer_utils import get_last_checkpoint

if not str(OUTPUT_DIR).startswith("gs://"):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

last_ckpt = get_last_checkpoint(OUTPUT_DIR)
if RESUME_TRAINING and last_ckpt is None:
    print(
        "No checkpoint-* under OUTPUT_DIR — starting from scratch (resume_from_checkpoint=None)."
    )
resume_arg = last_ckpt if (RESUME_TRAINING and last_ckpt) else None
print(f"OUTPUT_DIR={OUTPUT_DIR!r}  resume_from_checkpoint={resume_arg!r}")
trainer.train(resume_from_checkpoint=resume_arg)

Starting TPU training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Step,Training Loss,Validation Loss,Bleu
200,15.842465,11.393679,0.011438
400,9.519125,6.705601,0.018915


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 11. Save Final Model

In [ ]:
print("Saving model...")
trainer.save_model("final_model")
tokenizer.save_pretrained("final_model")

Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('final_model/tokenizer_config.json', 'final_model/tokenizer.json')

In [ ]:
DRIVE_MODEL_PATH = "/content/drive/MyDrive/models/encoder_decoder_llm"
print(f"Saving final model to Google Drive at: {DRIVE_MODEL_PATH}")
os.system(f"cp -r final_model {DRIVE_MODEL_PATH}")

Saving final model to Google Drive at: /content/drive/MyDrive/models/encoder_decoder_llm


0

## 12. Upload Checkpoints to GCS (Optional)

In [ ]:
import os

print(f"Uploading final model to {GCS_FINAL_MODEL_DIR} (creates models/final/ for next-run folder selection)")
os.system(f"gcloud storage cp -r final_model {GCS_FINAL_MODEL_DIR}")

print("Training complete.")

Uploading checkpoints to GCS...
Training complete.


## 13. Test Model with Custom Examples

In [ ]:
print("Loading saved model and tokenizer for inference...")

# Load the tokenizer and model from the saved directory
# If you saved to Google Drive, specify the DRIVE_MODEL_PATH here
# For simplicity, we'll assume 'final_model' is accessible locally after trainer.save_model()
loaded_tokenizer = AutoTokenizer.from_pretrained("final_model")
loaded_model = AutoModelForSeq2SeqLM.from_pretrained("final_model")

print("Model and tokenizer loaded successfully!")

Loading saved model and tokenizer for inference...


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model and tokenizer loaded successfully!


In [ ]:
print("Defining custom input examples...")

# Define some custom Romanized text inputs
custom_inputs = [
    "hello world",
    "egegnalehu",
    "salamu new",
    "yezare wulet",
    "ante sew endet new tefah eko"
]

print(f"Custom inputs: {custom_inputs}")

Defining custom input examples...
Custom inputs: ['hello world', 'egegnalehu', 'salamu new', 'yezare wulet', 'ante sew endet new tefah eko']


In [ ]:
import torch

print("Generating predictions...")

# Tokenize the custom inputs
inputs = loaded_tokenizer(custom_inputs, return_tensors="pt", padding=True, truncation=True, max_length=MAX_INPUT_LENGTH)

# Generate predictions
# Ensure the model is in evaluation mode if not already
loaded_model.eval()
with torch.no_grad(): # Use torch.no_grad() for inference to save memory and computations
    predictions = loaded_model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=MAX_TARGET_LENGTH
    )

# Decode the predictions
decoded_predictions = loaded_tokenizer.batch_decode(predictions, skip_special_tokens=True)

print("Predictions generated. Displaying results:")
for i, (input_text, predicted_text) in enumerate(zip(custom_inputs, decoded_predictions)):
    print(f"Input {i+1}: {input_text}")
    print(f"Predicted: {predicted_text}\n")

Generating predictions...
Predictions generated. Displaying results:
Input 1: hello world
Predicted: <extra_id_0>

Input 2: egegnalehu
Predicted: <extra_id_0>

Input 3: salamu new
Predicted: <extra_id_0>

Input 4: yezare wulet
Predicted: <extra_id_0>

Input 5: ante sew endet new tefah eko
Predicted: <extra_id_0>



### https://arxiv.org/pdf/2511.22769